## Imports
Load libraries for data processing and APIs.

In [62]:
import os                           # for environment variables (API key)
import warnings                     # suppress warnings
from pathlib import Path            # file path handling

import numpy as np                  # numerical operations
import pandas as pd                 # dataframes
import requests                     # API calls
import yfinance as yf              # Yahoo Finance data

warnings.filterwarnings("ignore")  # cleaner output
pd.set_option("display.max_columns", 100)

## Configuration
Define tickers and output paths.

In [63]:
START_DATE = "2010-01-01"
END_DATE = None

CCOPPER_TICKER = "HG=F"

FRED_SERIES = {
    "vix": "VIXCLS",
    "unrate": "UNRATE",
    "cpi": "CPIAUCSL",
    "fedfunds": "DFF",
    "indpro": "INDPRO",
    "dollar": "DTWEXBGS"
}

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_OUTPUT = OUTPUT_DIR / "copper_macro_raw_monthly.csv"
MODEL_OUTPUT = OUTPUT_DIR / "copper_macro_model_data.csv"

## Load API Key
Get FRED API key from environment (Streamlit will inject it).

In [64]:
FRED_API_KEY = os.getenv("FRED_API_KEY")
# get API key from environment (works with Streamlit)

if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY not found in environment")

print("API key loaded")

API key loaded


## FRED Helper Function
Download macroeconomic data.

In [65]:
def fred_series_observations(series_id, api_key, start_date="2010-01-01"):
    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "observation_start": start_date,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    data = r.json()["observations"]

    df = pd.DataFrame(data)[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.set_index("date").sort_index()

    s = df["value"]
    s.name = series_id

    return s

## Download Market Data
Get copper and VIX data.

In [66]:
market = yf.download(
    [COPPER_TICKER, VIX_TICKER],
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    group_by="column",
    progress=False
)

market.head()

print("market shape:", market.shape)
print(market.tail())

Price        Close               High                Low               Open  \
Ticker        HG=F       ^VIX    HG=F       ^VIX    HG=F       ^VIX    HG=F   
Date                                                                          
2010-01-04  3.3880  20.040001  3.4090  21.680000  3.3530  20.030001  3.3880   
2010-01-05  3.3960  19.350000  3.4110  20.129999  3.3685  19.340000  3.3960   
2010-01-06  3.4775  19.160000  3.4995  19.680000  3.4335  18.770000  3.4775   
2010-01-07  3.4115  19.059999  3.5235  19.709999  3.4110  18.700001  3.4995   
2010-01-08  3.3880  18.129999  3.4220  19.270000  3.3800  18.110001  3.3880   

Price                 Volume       
Ticker           ^VIX   HG=F ^VIX  
Date                               
2010-01-04  21.680000  404.0  0.0  
2010-01-05  20.049999  242.0  0.0  
2010-01-06  19.590000  109.0  0.0  
2010-01-07  19.680000  326.0  0.0  
2010-01-08  19.270000  311.0  0.0

## Convert to Monthly Data
Aggregate daily data.

In [67]:
copper_daily = market["Close"][COPPER_TICKER].rename("copper_price")
vix_daily = market["Close"][VIX_TICKER].rename("vix")

copper_monthly = copper_daily.resample("M").last()
vix_monthly = vix_daily.resample("M").mean()

market_monthly = pd.concat([copper_monthly, vix_monthly], axis=1).dropna()

market_monthly.head()

print("market_monthly shape:", market_monthly.shape)
print(market_monthly.tail())

,copper_price,vix
Date,,
2010-01-31,3.0460,20.643158
2010-02-28,3.2685,22.540000
2010-03-31,3.5460,17.767391
2010-04-30,3.3375,17.424286
2010-05-31,3.0970,31.929500


## Download Macro Data
Fetch FRED data.

In [68]:
fred_data = {}

for name, series_id in FRED_SERIES.items():
    fred_data[name] = fred_series_observations(
        series_id,
        FRED_API_KEY,
        START_DATE
    )

macro = pd.concat(fred_data.values(), axis=1)
macro.columns = fred_data.keys()

macro.head()

print("macro shape:", macro.shape)
print(macro.tail())

,unrate,cpi,fedfunds,indpro,dollar
date,,,,,
2010-01-01,9.8,217.488,0.05,89.3426,NaN
2010-01-02,NaN,NaN,0.05,NaN,NaN
2010-01-03,NaN,NaN,0.05,NaN,NaN
2010-01-04,NaN,NaN,0.12,NaN,92.3566
2010-01-05,NaN,NaN,0.12,NaN,92.2236


## Convert Macro Data
Align to monthly frequency.

In [69]:
macro_monthly = pd.DataFrame(index=market_monthly.index)

macro_monthly["unrate"] = macro["unrate"].resample("M").last()
macro_monthly["cpi"] = macro["cpi"].resample("M").last()
macro_monthly["indpro"] = macro["indpro"].resample("M").last()

macro_monthly["fedfunds"] = macro["fedfunds"].resample("M").mean()
macro_monthly["dollar"] = macro["dollar"].resample("M").mean()

macro_monthly.head()

print("macro_monthly shape:", macro_monthly.shape)
print(macro_monthly.tail())

,unrate,cpi,indpro,fedfunds,dollar
Date,,,,,
2010-01-31,9.8,217.488,89.3426,0.110000,92.440300
2010-02-28,9.8,217.281,89.6779,0.126429,93.886680
2010-03-31,9.9,217.353,90.2928,0.164516,93.149878
2010-04-30,9.9,217.403,90.5991,0.198333,92.639086
2010-05-31,9.6,217.290,91.8230,0.200645,95.530640


## Merge Data
Combine everything.

In [70]:
raw_monthly = pd.concat([market_monthly, macro_monthly], axis=1)

raw_monthly = raw_monthly.sort_index()
raw_monthly = raw_monthly.dropna()

raw_monthly.head()

print("raw_monthly shape:", raw_monthly.shape)
print(raw_monthly.tail())

,copper_price,vix,unrate,cpi,indpro,fedfunds,dollar
Date,,,,,,,
2010-01-31,3.0460,20.643158,9.8,217.488,89.3426,0.110000,92.440300
2010-02-28,3.2685,22.540000,9.8,217.281,89.6779,0.126429,93.886680
2010-03-31,3.5460,17.767391,9.9,217.353,90.2928,0.164516,93.149878
2010-04-30,3.3375,17.424286,9.9,217.403,90.5991,0.198333,92.639086
2010-05-31,3.0970,31.929500,9.6,217.290,91.8230,0.200645,95.530640


## Create Features
Generate predictive features.

In [71]:
df = raw_monthly.copy()

df["copper_ret_1m"] = df["copper_price"].pct_change()

df["copper_next_ret_1m"] = df["copper_price"].shift(-1) / df["copper_price"] - 1
df["copper_next_up"] = (df["copper_next_ret_1m"] > 0).astype(int)

df["cpi_yoy"] = df["cpi"].pct_change(12)
df["indpro_yoy"] = df["indpro"].pct_change(12)
df["dollar_yoy"] = df["dollar"].pct_change(12)

df["vix_3m_avg"] = df["vix"].rolling(3).mean()
df["vix_12m_z"] = (
    (df["vix"] - df["vix"].rolling(12).mean()) /
    df["vix"].rolling(12).std()
)

df["copper_mom_3m"] = df["copper_price"].pct_change(3)
df["copper_mom_6m"] = df["copper_price"].pct_change(6)

df.head()

print("df shape after features:", df.shape)
print(df.tail())

,copper_price,vix,unrate,cpi,indpro,fedfunds,dollar,copper_ret_1m,copper_next_ret_1m,copper_next_up,cpi_yoy,indpro_yoy,dollar_yoy,vix_3m_avg,vix_12m_z,copper_mom_3m,copper_mom_6m
Date,,,,,,,,,,,,,,,,,
2010-01-31,3.0460,20.643158,9.8,217.488,89.3426,0.110000,92.440300,NaN,0.073047,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-02-28,3.2685,22.540000,9.8,217.281,89.6779,0.126429,93.886680,0.073047,0.084901,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-03-31,3.5460,17.767391,9.9,217.353,90.2928,0.164516,93.149878,0.084901,-0.058799,0,NaN,NaN,NaN,20.316850,NaN,NaN,NaN
2010-04-30,3.3375,17.424286,9.9,217.403,90.5991,0.198333,92.639086,-0.058799,-0.072060,0,NaN,NaN,NaN,19.243892,NaN,0.095699,NaN
2010-05-31,3.0970,31.929500,9.6,217.290,91.8230,0.200645,95.530640,-0.072060,-0.051986,0,NaN,NaN,NaN,22.373726,NaN,-0.052471,NaN


## Prepare Model Dataset
Select final columns.

In [72]:
model_data = df[[
    "copper_price",
    "copper_ret_1m",
    "copper_next_ret_1m",
    "copper_next_up",
    "vix",
    "vix_3m_avg",
    "vix_12m_z",
    "unrate",
    "cpi_yoy",
    "fedfunds",
    "indpro_yoy",
    "dollar_yoy",
    "copper_mom_3m",
    "copper_mom_6m"
]].dropna()

model_data.head()

print("model_data shape:", model_data.shape)
print(model_data.tail())
print("NaNs per column:\n", model_data.isna().sum())

,copper_price,copper_ret_1m,copper_next_ret_1m,copper_next_up,vix,vix_3m_avg,vix_12m_z,unrate,cpi_yoy,fedfunds,indpro_yoy,dollar_yoy,copper_mom_3m,copper_mom_6m
Date,,,,,,,,,,,,,,
2011-01-31,4.4510,0.002590,0.006066,1,17.315500,18.326920,-1.013312,9.1,0.017008,0.168387,0.046325,-0.025196,0.192658,0.345933
2011-02-28,4.4780,0.006066,-0.039750,0,17.430000,17.438348,-0.869324,9.0,0.021249,0.156786,0.038274,-0.047824,0.171331,0.332342
2011-03-31,4.3000,-0.039750,-0.031279,0,20.723478,18.489659,-0.283181,9.0,0.026192,0.138710,0.041930,-0.050055,-0.031422,0.179375
2011-04-30,4.1655,-0.031279,0.001921,1,16.244000,18.132493,-1.137284,9.1,0.030772,0.098000,0.035053,-0.061720,-0.064143,0.116158
2011-05-31,4.1735,0.001921,0.023601,1,16.911429,17.959636,-0.920173,9.0,0.034590,0.093871,0.022489,-0.089858,-0.067999,0.091682


## Save Data
Save outputs for modeling.

In [73]:
raw_monthly.to_csv(RAW_OUTPUT)
model_data.to_csv(MODEL_OUTPUT)

print("Data saved successfully")

Data saved successfully
